### this is for the visual only session 20240821_121447

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from labdata.schema import UnitCount, UnitMetrics, UnitCountCriteria, SpikeSorting

pd.set_option("display.max_columns", 100)

plt.rcParams["text.usetex"] = False
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["font.size"] = 12
plt.rcParams["figure.dpi"] = 100

%config InlineBackend.figure_format = 'svg'
%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
visual_session = "20240821_121447"

In [ ]:
UnitMetrics() & f'session_name = "{visual_session}"'

In [ ]:
params = {
    "unit_criteria_id": 1,
    "sua_criteria": "isi_contamination < 0.1 & amplitude_cutoff < 0.1 & spike_duration > 0.1 & spike_amplitude > 50 & presence_ratio > 0.6 & firing_rate > 1",
}
UnitCountCriteria.insert(rows=[params], skip_duplicates=True)
UnitCountCriteria()

In [ ]:
(
    UnitCount.Unit() * UnitMetrics()
) & "passes = 1" & f'session_name = "{visual_session}"' & "unit_criteria_id = 1"

In [ ]:
good_units = (
    UnitCount.Unit()
    & "passes = 1"
    & f'session_name = "{visual_session}"'
    & "unit_criteria_id = 1"
)

In [ ]:
SpikeSorting.Unit() * UnitMetrics() * good_units

In [ ]:
unit_id, spike_times, depth = (SpikeSorting.Unit() * UnitMetrics() * good_units).fetch(
    "unit_id", "spike_times", "depth"
)

pd.DataFrame({"unit_id": unit_id, "spike_times": spike_times, "depth":depth}).to_pickle(
    f"/Users/gabriel/Downloads/{visual_session}_ks4_spike_times.pkl"
)

In [ ]:
double_peaks = {
    1: "short isi",
    57: "good",
    111: "good",
    147: "short isi",
    145: "short isi",
    182: "short isi",
}

weird = {5: "short isi", 86: "short isi", 97: "good", 101: "good", 52: "short isi"}

single_peaks = {
    43: "short isi",
    66: "bimodal distribution of isi",
    150: "short isi",
    82: "short isi",
    108: "good",
    95: "really short isi",
    3: "short isi",
    119: "good",
    166: "good",
    140: "good",
    180: "really short isi",
}

notes = {"double_peaks": double_peaks, "weird": weird, "single_peaks": single_peaks}

double_clusters = unit_id[list(notes["double_peaks"].keys())]
weird_clusters = unit_id[list(notes["weird"].keys())]
single_clusters = unit_id[list(notes["single_peaks"].keys())]

In [ ]:
clu_ids = []
for idx in pd.DataFrame(notes).index:
    clu_ids.append(unit_id[idx])

In [ ]:
data = pd.DataFrame(
    UnitCount.Unit() * SpikeSorting.Unit() * UnitMetrics()
    & f'session_name = "{visual_session}"'
    & "unit_criteria_id = 1"
    & f"unit_id IN {tuple(clu_ids)}"
)

In [ ]:
plt.figure()
plt.scatter(data.spike_duration, data.depth)
plt.xlabel("spike duration")
plt.ylabel("depth")
plt.title("good units")
plt.tight_layout()

### good units

In [ ]:
plt.figure()
plt.hist(data.spike_duration, bins=30)
plt.xlabel("spike duration")
plt.ylabel("count")
plt.title("good units")
plt.tight_layout()

In [ ]:
plt.figure()
plt.hist(
    data[
        data.unit_id.isin(np.concatenate((double_clusters, weird_clusters)))
    ].spike_duration,
    bins=10,
    label="double",
    alpha=0.8,
)
plt.hist(
    data[data.unit_id.isin(single_clusters)].spike_duration,
    bins=10,
    label="single",
    alpha=0.8,
)
plt.xlabel("spike duration")
plt.ylabel("count")
plt.title("good units")
plt.legend()
plt.tight_layout()

In [ ]:
plt.figure()
plt.scatter(
    data[
        data.unit_id.isin(np.concatenate((double_clusters, weird_clusters)))
    ].spike_duration,
    data[
        data.unit_id.isin(np.concatenate((double_clusters, weird_clusters)))
    ].spike_amplitude,
    label="double+weird",
)
plt.scatter(
    data[data.unit_id.isin(single_clusters)].spike_duration,
    data[data.unit_id.isin(single_clusters)].spike_amplitude,
    label="single",
)
plt.xlabel("spike_duration")
plt.ylabel("spike_amplitude")
plt.legend();

In [ ]:
plt.figure()
plt.scatter(
    data[
        data.unit_id.isin(np.concatenate((double_clusters, weird_clusters)))
    ].spike_duration,
    data[
        data.unit_id.isin(np.concatenate((double_clusters, weird_clusters)))
    ].firing_rate,
    label="double+weird",
)
plt.scatter(
    data[data.unit_id.isin(single_clusters)].spike_duration,
    data[data.unit_id.isin(single_clusters)].firing_rate,
    label="single",
)
plt.xlabel("spike_duration")
plt.ylabel("firing_rate")
plt.legend();

### all units

In [ ]:
all_data = pd.DataFrame(
    (UnitCount.Unit() * SpikeSorting.Unit() * UnitMetrics())
    & f'session_name = "{visual_session}"'
    & "passes = 1"
    & "unit_criteria_id = 1"
)

In [ ]:
plt.figure()
plt.hist(all_data.spike_duration, bins=30, alpha=0.8)
plt.xlabel("spike_duration")
plt.ylabel("count")
plt.title("all units")
plt.tight_layout()

In [ ]:
plt.figure()
plt.scatter(all_data.spike_duration, all_data.firing_rate, alpha=0.5)
plt.xlabel("spike_duration")
plt.ylabel("spike_amplitude")

In [ ]:
plt.figure()

is_double = all_data.unit_id.isin(np.concatenate((double_clusters, weird_clusters)))

plt.scatter(
    all_data.loc[~is_double, "spike_duration"],
    all_data.loc[~is_double, "firing_rate"],
    alpha=0.5,
    label="other",
)

plt.scatter(
    all_data.loc[is_double, "spike_duration"],
    all_data.loc[is_double, "firing_rate"],
    alpha=0.5,
    c="tab:orange",
    label="double peak",
)

plt.xlabel("spike_duration")
plt.ylabel("firing_rate")
plt.legend()
plt.tight_layout()

In [ ]:
plt.figure()

is_double = all_data.unit_id.isin(np.concatenate((double_clusters, weird_clusters)))

plt.scatter(
    all_data.loc[~is_double, "spike_duration"],
    all_data.loc[~is_double, "depth"],
    alpha=0.5,
    label="other",
)

plt.scatter(
    all_data.loc[is_double, "spike_duration"],
    all_data.loc[is_double, "depth"],
    alpha=0.5,
    c="tab:orange",
    label="double peak",
)

plt.xlabel("spike_duration")
plt.ylabel("depth")
plt.legend()
plt.tight_layout()

In [ ]:
is_double.rename("double")

In [ ]:
ad = pd.concat([all_data, is_double.rename("is_double")], axis=1)

In [ ]:
import seaborn as sns

plt.figure()
sns.violinplot(data=ad, hue="is_double", x="depth", fill=False, bw_adjust=0.5)

In [ ]:
a = [4, 20]
b = [0.2, 0.8]
plt.figure(figsize=(4, 4))
plt.plot(a, b)
plt.xlabel("stimulus frequency (Hz)")
plt.ylabel("p(R) choice")
plt.xticks([4, 20])
plt.yticks([0, 0.5, 1])
plt.tight_layout()

# Survey map 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display


plt.rcParams["text.usetex"] = False
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["font.size"] = 12
plt.rcParams["figure.dpi"] = 100

%config InlineBackend.figure_format = 'svg'
%matplotlib widget
%load_ext autoreload
%autoreload 2

In [5]:
survey_map = pd.read_csv(
    "/Users/gabriel/Downloads/Organized/Spreadsheets/GRB058_SvyPrb_map_2026-01-29.csv"
)

pal = "rocket_r"
vmin = 0
vmax = 1

vmax_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=60,
    step=5,
    description="vmax",
)

zum_slider = widgets.IntRangeSlider(
    value=(0, 5000),
    min=int(survey_map["Zum"].min()),
    max=int(survey_map["Zum"].max()),
    step=1000,
    description="Zum",
    orientation="vertical",
)

out = widgets.Output()
with out:
    plt.ioff()
    fig, ax = plt.subplots()
    plt.ion()  # Re-enable if needed for other cells

    sm = mpl.cm.ScalarMappable(
        norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax), cmap=pal
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, label="norm channel voltage")
    display(fig.canvas)


def draw(*_):
    vmax = float(vmax_slider.value)
    zmin, zmax = map(int, zum_slider.value)

    df = survey_map[(survey_map["Zum"] >= zmin) & (survey_map["Zum"] <= zmax)]

    ax.clear()
    sns.stripplot(
        data=df,
        x="Shank",
        y="Zum",
        hue="Val",
        hue_norm=(vmin, vmax),
        palette=pal,
        size=3.5,
        alpha=0.5,
        ax=ax,
        legend=False,
    )

    ax.set_xlabel("shanks (M->L)")
    ax.set_ylabel("depth from probe tip (µm)")

    # Update colorbar limits mapping
    sm.set_clim(vmin, vmax)

    fig.tight_layout()
    fig.canvas.draw_idle()


vmax_slider.observe(draw, names="value")
zum_slider.observe(draw, names="value")

# Rearrange layout: vmax slider on top, zum slider next to plot
display(widgets.VBox([vmax_slider, widgets.HBox([zum_slider, out])]))

draw()